# RadiScan — training on Kaggle

Runs the **real** pipeline from the repo, not a reimplementation: same dataset
class, same losses, same trainer, same evaluation. Whatever this produces is
reproducible on your laptop with the same commands.

## Before you run anything

In the right-hand panel:

1. **Accelerator → GPU P100** (or T4 x2 — the notebook probes whichever it gets)
2. **Internet → On** — required to reach GitHub and fetch the ImageNet weights
3. **Input → Add Input** → search **CheXpert** and attach the small version

Session limits: 12 h per run, 30 h/week of GPU. A 25-epoch run on 40k images
takes 1–2 h, so there is plenty of headroom.


## 1. Environment

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
# Clone the repo. Public, so no token needed.
#
# NOT into /kaggle/working/radiscan: Kaggle puts /kaggle/working on sys.path, so
# a directory named `radiscan` there becomes an implicit namespace package
# (PEP 420) and shadows the real package in src/. `import radiscan` then
# succeeds but hands back an empty module -- no __version__, no submodules.
!rm -rf /kaggle/working/repo
!git clone -q https://github.com/vanshvj01/RadiScan-AI.git /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -1

In [ ]:
# Kaggle ships torch already, so install only what is missing and never let pip
# touch torch -- replacing it mid-session breaks the CUDA build.
!pip install -q timm albumentations pydicom pydantic-settings 2>&1 | tail -2
!pip install -q --no-deps -e . 2>&1 | tail -2

# Kaggle's Python 3.13 kernel does not see a package installed with
# `pip install -e .` until the kernel restarts, so `import radiscan` fails with
# ModuleNotFoundError right after a successful install. Put the source on the
# path directly, for this kernel and for every `!python -m ...` subprocess.
import importlib, os, sys
SRC = "/kaggle/working/repo/src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)
# If anything scanned SRC before the clone created it, the import system cached
# it as empty. Drop that cache entry so the import below looks again.
sys.path_importer_cache.pop(SRC, None)
importlib.invalidate_caches()
os.environ["PYTHONPATH"] = f"{SRC}:/kaggle/working/repo:" + os.environ.get("PYTHONPATH", "")

import radiscan

# Assert we imported the real package rather than a namespace-package shadow.
# Without this the failure resurfaces later as a confusing missing-submodule
# error somewhere deep in training.
assert hasattr(radiscan, "__version__"), (
    f"Imported a shadowed 'radiscan' from {getattr(radiscan, '__path__', '?')}. "
    "Restart the session (Run -> Restart & Clear Cell Outputs) and re-run."
)
print("radiscan", radiscan.__version__, "from", radiscan.__file__)

## 2. Find the dataset

Kaggle mounts each attached dataset under `/kaggle/input/<slug>/`, and the slug
differs between CheXpert mirrors. Rather than hardcoding a path that breaks when
you attach a different copy, this searches for `train.csv` and works out the
root from where the image folders actually are.

In [ ]:
from pathlib import Path

candidates = sorted(Path("/kaggle/input").rglob("train.csv"))
print("train.csv found at:")
for c in candidates:
    print("  ", c)
assert candidates, "No train.csv under /kaggle/input -- did you attach the dataset?"

TRAIN_CSV = candidates[0]
CSV_DIR = TRAIN_CSV.parent
print("\nUsing:", TRAIN_CSV)

In [ ]:
import pandas as pd

df = pd.read_csv(TRAIN_CSV, nrows=5)
sample_rel = str(df["Path"].iloc[0])
print("rows look like:", sample_rel)
print("columns:", len(pd.read_csv(TRAIN_CSV, nrows=0).columns))

# The Path column may or may not carry the "CheXpert-v1.0-small/" prefix, and
# the images may sit beside the CSV or one level up. Try both and keep whichever
# actually resolves -- guessing here is what produces a run that trains happily
# on nothing.
DATA_ROOT = None
STRIP = False
for root in [CSV_DIR, CSV_DIR.parent, *CSV_DIR.parent.iterdir()]:
    if not root.is_dir():
        continue
    if (root / sample_rel).exists():
        DATA_ROOT, STRIP = root, False
        break
    stripped = sample_rel.split("/", 1)[-1]
    if (root / stripped).exists():
        DATA_ROOT, STRIP = root, True
        print("NOTE: stripping the leading folder from Path entries")
        break

assert DATA_ROOT, f"Could not resolve {sample_rel!r} under {CSV_DIR} or its parent"
print("\nDATA_ROOT :", DATA_ROOT)
print("strip prefix:", STRIP)

In [ ]:
# If the CSV carries a prefix the images do not have, rewrite the CSV once
# rather than special-casing it everywhere downstream.
WORK = Path("/kaggle/working/data")
WORK.mkdir(parents=True, exist_ok=True)

full = pd.read_csv(TRAIN_CSV)
if STRIP:
    full["Path"] = full["Path"].str.split("/", n=1).str[-1]
    print("stripped leading folder from", len(full), "paths")

MASTER_CSV = WORK / "master.csv"
full.to_csv(MASTER_CSV, index=False)
print("master CSV:", MASTER_CSV, "|", len(full), "rows")
print("spot check :", (DATA_ROOT / str(full["Path"].iloc[0])).exists())

## 3. Patient-disjoint splits

CheXpert has several studies per patient. A random row split puts the same
patient in train and validation, and the resulting AUC is inflated by 3–6
points. This splits on patient ID and asserts zero overlap.

In [ ]:
!python -m radiscan.cli.prepare_data \
    --source {MASTER_CSV} \
    --out-dir /kaggle/working/data/splits \
    --frontal-only \
    --train-size 40000 --val-size 4000 --test-size 4000

## 4. Write the config against the discovered paths

In [ ]:
cfg_text = f"""data:
  dataset: chexpert
  data_dir: {DATA_ROOT}
  train_csv: /kaggle/working/data/splits/train.csv
  val_csv: /kaggle/working/data/splits/val.csv
  test_csv: /kaggle/working/data/splits/test.csv
  image_size: 320
  uncertain_policy: lsr
  num_workers: 2
  balanced_sampling: true
  frontal_only: true
model:
  backbone: densenet121
  pretrained: true
  dropout: 0.2
loss:
  kind: asl
train:
  epochs: 25
  batch_size: 32
  accumulate_steps: 1
  lr: 0.0001
  head_lr_mult: 10.0
  warmup_epochs: 2
  scheduler: cosine
  amp: true
  ema_decay: 0.999
  freeze_backbone_epochs: 1
  mixup_alpha: 0.2
  early_stopping_patience: 6
  seed: 42
  checkpoint_dir: /kaggle/working/checkpoints
  log_dir: /kaggle/working/logs
inference:
  mc_dropout_samples: 20
  tta_hflip: true
"""
CFG = Path("/kaggle/working/kaggle.yaml")
CFG.write_text(cfg_text)
print(cfg_text)

### Size the batch to whatever GPU you were given

P100 and T4 both have 16 GB, but the probe is cheap and removes the guess. It
also refuses batch sizes that only "fit" by spilling into host memory, which on
some drivers happens silently instead of raising.

In [ ]:
# Probe VRAM and write the recommendation to a file, so the training cell picks
# it up automatically. No manual step between here and training.
!python scripts/vram_probe.py --config {CFG} --max-batch 128 | tee /kaggle/working/probe.txt

import re
from pathlib import Path

txt = Path("/kaggle/working/probe.txt").read_text()
m = re.search(r"batch_size:\s*(\d+)", txt)
BATCH = int(m.group(1)) if m else 32
print(f"\n>>> training will use batch_size={BATCH}")

## 5. Verify before burning GPU hours

Three seconds. Checks that CSV paths resolve to real files and prints the
per-class positive rates. If any class reads 0%, stop and fix it — that is a
column-name mismatch, and training would silently optimise nothing.

In [ ]:
!python -m radiscan.cli.verify_data --config {CFG}

## 6. Train

25 epochs on 40k images is roughly 1–2 h. A checkpoint is written every epoch to
`/kaggle/working/checkpoints/`, so a disconnect costs one epoch rather than the
run.

Afterwards it automatically fits a temperature and tunes per-class thresholds,
and writes both into the checkpoint.

In [ ]:
!python -m radiscan.cli.train --config {CFG} --set train.batch_size={BATCH}

## 7. Evaluate on the held-out test split

Produces every table and figure the report needs. Note `--split test` — the
validation set was used to tune thresholds and temperature, so quoting
validation numbers as your headline result is not defensible.

In [ ]:
!python -m radiscan.cli.evaluate --config {CFG} --split test --out /kaggle/working/results

In [ ]:
import pandas as pd
pd.read_csv("/kaggle/working/results/per_class_metrics.csv")[
    ["label", "support", "auc", "ap", "precision", "recall", "f1"]
].round(4)

In [ ]:
from IPython.display import Image, display
for name in ["per_class_auc", "roc_curves", "pr_curves", "calibration",
             "uncertainty_quartiles", "selective_prediction", "training_curves"]:
    p = f"/kaggle/working/results/{name}.png"
    if Path(p).exists():
        print(name); display(Image(p))

### The table that justifies the uncertainty work

If accuracy falls monotonically Q1 → Q4, the MC-Dropout estimates carry real
signal and the entire uncertainty half of the project is validated in one table.
If it does not, report that honestly and say why you think so — a clean negative
result reads far better in a viva than a fudged positive one.

In [ ]:
import json
s = json.load(open("/kaggle/working/results/summary.json"))
print("Mean AUC (14)        :", round(s["overall"]["mean_auc"], 4))
print("Mean AP              :", round(s["overall"]["mean_ap"], 4))
print("Mean AUC (CheXpert-5):", round(s["chexpert_competition_5"]["mean_auc"], 4))
print("ECE                  :", round(s["ece"], 4))
print("Uncertainty informative:", s["uncertainty_is_informative"])
print()
pd.read_csv("/kaggle/working/results/uncertainty_quartiles.csv")

## 8. Take it home

Everything under `/kaggle/working/` is downloadable from the notebook's **Output**
tab once the session ends. The zip below bundles the checkpoint and every result
so it is one download rather than thirty.

Drop `best_model.pth` into `models/checkpoints/` on your laptop and the API and
dashboard serve it immediately — inference is fine on your 4050, or even on CPU.

In [ ]:
import shutil, json
from pathlib import Path

s = json.load(open("/kaggle/working/results/summary.json"))
auc = s["overall"]["mean_auc"]

out = Path("/kaggle/working/trained_bundle")
out.mkdir(exist_ok=True)
shutil.copy("/kaggle/working/checkpoints/best_model.pth", out)
shutil.copytree("/kaggle/working/results", out / "results", dirs_exist_ok=True)
shutil.copytree("/kaggle/working/logs", out / "logs", dirs_exist_ok=True)
shutil.copy(CFG, out)

shutil.make_archive("/kaggle/working/trained_bundle", "zip", out)
size = Path("/kaggle/working/trained_bundle.zip").stat().st_size / 1e6
print(f"trained_bundle.zip  ({size:.0f} MB)  mean AUC {auc:.4f}")
print("\nDownload it from the Output tab, then on your laptop:")
print("  copy best_model.pth  ->  models\\checkpoints\\")
print("  uvicorn services.api.main:app --reload")